In [2]:
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("SilverLayerTransformations") \
    .enableHiveSupport() \
    .getOrCreate()

In [3]:
# This is for clean formatting when showing the dataframe

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 60)

In [4]:
spark.sql("SHOW DATABASES").show()

2026-09-05 02:35:58,318 WARN conf.HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
2026-09-05 02:35:58,319 WARN conf.HiveConf: HiveConf of name hive.stats.retries.wait does not exist


+------------+
|   namespace|
+------------+
|churn_bronze|
|     default|
|   retail_db|
+------------+



In [5]:
spark.sql("USE churn_bronze")

2026-09-05 02:36:01,841 WARN metastore.ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


""


In [6]:
df_customers = spark.read.format("avro").load("hdfs://localhost:9000/bronze/customers")

df_customers.printSchema()
df_customers

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- geography: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- tenure_months: string (nullable = true)
 |-- date_opened: string (nullable = true)
 |-- last_modified: string (nullable = true)



customer_id,name,email,age,gender,geography,is_active,tenure_months,date_opened,last_modified
CUST-100000,Nicole Bernard,marissavang@examp...,37,Other,"Cairo, EG",FALSE,22,2023-11-10,2024-06-18 04:45:...
CUST-100001,April Anderson,richard.brooks@en...,24,FEMALE,null,Y,59,2020/10/14,2024-07-15 20:25:...
CUST-100002,Karen Martinez,alexander03@examp...,72,F,null,True,44,2022-01-14,2025-02-01 23:33:...
CUST-100003,John Garza,KATHERINE40@EXAMP...,74,Other,"Alexandria, EG",TRUE,47,11-Oct-2021,2025-04-24 06:34:...
CUST-100004,Tyler Lyons,ashley.mercer@com...,999,Other,"Dubai, AE",no,16,2024/05/05,2024-07-18 08:34:...
CUST-100005,Timothy Rivera,sharon.morrison@e...,24,M,"Toronto, CA",yes,20,2024-01-17,2025-08-19 11:19:...
CUST-100006,Andrew Cooper,david.fernandez@t...,74,FEMALE,"Riyadh, SA",true,7,2025/02/14,2024-12-11 02:39:...
CUST-100007,Christopher Martin,henryraymond@exam...,26,Other,"Riyadh, SA",TRUE,55,2021/02/04,2024-06-10 04:09:...
CUST-100008,Terry Williams,hodgessean@exampl...,25,Other,"Riyadh, SA",TRUE,35,09/10/2022,2024-07-10 05:18:...
CUST-100009,Alexander Robertson,brady16@example.org,20,female,"Sharqia, EG",Y,59,23/10/2020,2025-03-05 17:53:...


In [18]:
# Counting nulls per column
df_customers.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df_customers.columns
]).show()

df_customers.count()

# It is safe to drop null age columns and last_modified
# Gender and geography can be imputed to "Unknown"
# tenure_months will be defaulted to 0 if null 

+-----------+----+-----+---+------+---------+---------+-------------+-----------+-------------+
|customer_id|name|email|age|gender|geography|is_active|tenure_months|date_opened|last_modified|
+-----------+----+-----+---+------+---------+---------+-------------+-----------+-------------+
|          0|   0|    0|798|  1934|     1283|        0|          576|          0|           71|
+-----------+----+-----+---+------+---------+---------+-------------+-----------+-------------+



19529

In [61]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, BooleanType, TimestampType

# 1. Clean and Transform Customers Bronze Data
customers_silver = (
    df_customers
    .filter(
        F.col("customer_id").isNotNull() & 
        (F.trim(F.col("customer_id")) != "")
    )
    # Trim identifiers and standard text fields
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    
    .withColumn(
        "name", 
        F.initcap(F.coalesce(F.trim(F.col("name")), F.lit("Unknown")))
    )
    
    .withColumn("email", F.coalesce(F.lower(F.trim(F.col("email"))), F.lit("unknown@domain.com")))
    
    # Cast age and DROP rows with NULL or invalid values (< 18 or > 100)
    .withColumn("age", F.col("age").cast(IntegerType()))
    .filter(
        F.col("age").isNotNull() & 
        (F.col("age") >= 18) & 
        (F.col("age") <= 100)
    )
    
    # Normalize gender to standard values (Fills nulls with 'Unknown')
    .withColumn(
        "gender",
        F.when(F.lower(F.col("gender")).isin("female", "f"), "Female")
         .when(F.lower(F.col("gender")).isin("male", "m"), "Male")
         .when(F.lower(F.col("gender")) == "other", "Other")
         .otherwise("Unknown")
    )
    
    # Standardize geography (Replaces SQL null strings like '\N' and blank values with 'Unknown')
    .withColumn(
        "geography",
        F.when(
            F.col("geography").isNull() | 
            (F.col("geography") == "\\N") | 
            (F.trim(F.col("geography")) == ""),
            "Unknown"
        ).otherwise(F.trim(F.col("geography")))
    )
    
    # Cast is_active string flags to True/False Boolean
    .withColumn(
        "is_active",
        F.when(F.lower(F.col("is_active")).isin("true", "t", "yes", "y", "1", "active"), True)
         .when(F.lower(F.col("is_active")).isin("false", "f", "no", "n", "0", "inactive"), False)
         .otherwise(False)
         .cast(BooleanType())
    )
    
    # Cast numeric metrics, defaulting NULL tenure to 0
    .withColumn("tenure_months", F.coalesce(F.col("tenure_months").cast(IntegerType()), F.lit(0)))
    .filter(
        F.col("tenure_months") >= 0
    )
    
    # Parse mixed date string formats into standard DateType (YYYY-MM-DD)
    .withColumn(
        "date_opened",
        F.coalesce(
            F.to_date(F.col("date_opened"), "yyyy-MM-dd"),
            F.to_date(F.col("date_opened"), "yyyy/MM/dd"),
            F.to_date(F.col("date_opened"), "dd-MMM-yyyy"),
            F.to_date(F.col("date_opened"), "dd/MM/yyyy"),
            F.to_date(F.col("date_opened"), "MM-dd-yyyy"),
        )
    )
    
    # Drop unknown date opened
    .filter(F.col("date_opened").isNotNull())
    
    # Cast audit timestamp and dropping unknown dates
    .withColumn("last_modified", F.col("last_modified").cast(TimestampType()))
    .filter(F.col("last_modified").isNotNull())
    
    .dropDuplicates(["customer_id"])
)

In [62]:
customers_silver.printSchema()
customers_silver


root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = false)
 |-- email: string (nullable = false)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = false)
 |-- geography: string (nullable = true)
 |-- is_active: boolean (nullable = false)
 |-- tenure_months: integer (nullable = false)
 |-- date_opened: date (nullable = true)
 |-- last_modified: timestamp (nullable = true)



+-----------+----+-----+---+------+---------+---------+-------------+-----------+-------------+
|customer_id|name|email|age|gender|geography|is_active|tenure_months|date_opened|last_modified|
+-----------+----+-----+---+------+---------+---------+-------------+-----------+-------------+
|          0|   0|    0|  0|     0|        0|        0|            0|          0|            0|
+-----------+----+-----+---+------+---------+---------+-------------+-----------+-------------+



In [7]:
# Tickets

df_tickets = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("hdfs://localhost:9000/bronze/tickets")
)


# Print schema and row count
df_tickets.printSchema()
print("Total Ticket Records:", df_tickets.count())

# Inspect null counts per column
df_tickets.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df_tickets.columns
]).show()

df_tickets

root
 |-- Ticket_ID: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Customer_Email: string (nullable = true)
 |-- Ticket_Subject: string (nullable = true)
 |-- Ticket_Description: string (nullable = true)
 |-- Issue_Category: string (nullable = true)
 |-- Priority_Level: string (nullable = true)
 |-- Ticket_Channel: string (nullable = true)
 |-- Submission_Date: string (nullable = true)
 |-- Resolution_Time_Hours: integer (nullable = true)
 |-- Assigned_Agent: string (nullable = true)
 |-- Satisfaction_Score: integer (nullable = true)

Total Ticket Records: 20000
+---------+-----------+-------------+--------------+--------------+------------------+--------------+--------------+--------------+---------------+---------------------+--------------+------------------+
|Ticket_ID|customer_id|Customer_Name|Customer_Email|Ticket_Subject|Ticket_Description|Issue_Category|Priority_Level|Ticket_Channel|Submission_Date|Resoluti

Ticket_ID,customer_id,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,Assigned_Agent,Satisfaction_Score
TKT-100000,CUST-105976,George Simon,lisastrickland@ex...,Hours of operatio...,"Hi Support, Where...",General Inquiry,High,Web Form,2025-07-02,43,David Kim,5
TKT-100001,CUST-113634,Scott Thompson,wevans@example.org,Data not syncing ...,"Hi Support, The a...",Technical,HIGH,Chat,2025-06-28,41,Elena Rodriguez,5
TKT-100002,CUST-115106,Jennifer Smith,oleonard@example.net,2FA issues - Ques...,"Hi Support, How d...",Account,High,Web Form,2025-02-05,7,Anya Sharma,5
TKT-100003,CUST-113219,Rachel Bullock,katherine67@examp...,Login failed - Let,"Hi Support, The d...",Technical,LOW,Web Form,2025-03-20,41,Anya Sharma,5
TKT-100004,CUST-114054,Thomas Parks DDS,raykelsey@example...,Refund status - A...,"Hi Support, I hav...",Billing,Medium,Email,2025-04-27,40,David Kim,5
TKT-100005,CUST-106122,Ashley Stewart,gibsonrose@exampl...,Office location -...,"Hi Support, Where...",General Inquiry,Medium,Chat,2025-10-02,23,Elena Rodriguez,4
TKT-100006,CUST-104501,Ashley Bennett,gfox@example.org,Password reset - ...,"Hi Support, How d...",Account,Medium,Chat,2024-09-28,106,David Kim,3
TKT-100007,CUST-102680,Travis Blanchard,francismegan@exam...,Payment failed - Win,"Hi Support, My su...",Billing,Medium,Email,2024-10-26,27,David Kim,1
TKT-100008,CUST-109625,Lisa Thornton,Ashley.Mccarthy@t...,Login failed - Son,"Hi Support, I can...",Technical,High,Web Form,2025-04-29,65,David Kim,4
TKT-100009,CUST-107649,Richard Brewer,hjacobson@example...,Product question ...,"Hi Support, Is th...",General Inquiry,Low,Web Form,2024-06-30,110,Ben Carter,3


In [47]:
df_tickets.show(10, truncate=False)

#lowercase the column names just like customers table
#ticket_id must drop duplicates and nulls
#customer_id must drop nulls
#customer_name needs to be formatted (trim whitespaces, capitalize) and null handled
#Ticket description has a weird sentence after the main ticket subject that needs to removed.
df_tickets

+----------+-----------+----------------+--------------------------+-------------------------------+--------------------------------------------------------------------------------------------------------------+---------------+--------------+--------------+---------------+---------------------+---------------+------------------+
|Ticket_ID |customer_id|Customer_Name   |Customer_Email            |Ticket_Subject                 |Ticket_Description                                                                                            |Issue_Category |Priority_Level|Ticket_Channel|Submission_Date|Resolution_Time_Hours|Assigned_Agent |Satisfaction_Score|
+----------+-----------+----------------+--------------------------+-------------------------------+--------------------------------------------------------------------------------------------------------------+---------------+--------------+--------------+---------------+---------------------+---------------+------------------+
|TKT-10

Ticket_ID,customer_id,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,Assigned_Agent,Satisfaction_Score
TKT-100000,CUST-105976,George Simon,lisastrickland@ex...,Hours of operatio...,"Hi Support, Where...",General Inquiry,High,Web Form,2025-07-02,43,David Kim,5
TKT-100001,CUST-113634,Scott Thompson,wevans@example.org,Data not syncing ...,"Hi Support, The a...",Technical,HIGH,Chat,2025-06-28,41,Elena Rodriguez,5
TKT-100002,CUST-115106,Jennifer Smith,oleonard@example.net,2FA issues - Ques...,"Hi Support, How d...",Account,High,Web Form,2025-02-05,7,Anya Sharma,5
TKT-100003,CUST-113219,Rachel Bullock,katherine67@examp...,Login failed - Let,"Hi Support, The d...",Technical,LOW,Web Form,2025-03-20,41,Anya Sharma,5
TKT-100004,CUST-114054,Thomas Parks DDS,raykelsey@example...,Refund status - A...,"Hi Support, I hav...",Billing,Medium,Email,2025-04-27,40,David Kim,5
TKT-100005,CUST-106122,Ashley Stewart,gibsonrose@exampl...,Office location -...,"Hi Support, Where...",General Inquiry,Medium,Chat,2025-10-02,23,Elena Rodriguez,4
TKT-100006,CUST-104501,Ashley Bennett,gfox@example.org,Password reset - ...,"Hi Support, How d...",Account,Medium,Chat,2024-09-28,106,David Kim,3
TKT-100007,CUST-102680,Travis Blanchard,francismegan@exam...,Payment failed - Win,"Hi Support, My su...",Billing,Medium,Email,2024-10-26,27,David Kim,1
TKT-100008,CUST-109625,Lisa Thornton,Ashley.Mccarthy@t...,Login failed - Son,"Hi Support, I can...",Technical,High,Web Form,2025-04-29,65,David Kim,4
TKT-100009,CUST-107649,Richard Brewer,hjacobson@example...,Product question ...,"Hi Support, Is th...",General Inquiry,Low,Web Form,2024-06-30,110,Ben Carter,3


In [55]:
from pyspark.sql.types import DoubleType

tickets_silver = (
    df_tickets
    # 1. Drop records missing primary keys or customer links
    .filter(
        F.col("Ticket_ID").isNotNull() & (F.trim(F.col("Ticket_ID")) != "") &
        F.col("customer_id").isNotNull() & (F.trim(F.col("customer_id")) != "")
    )
    
    # 2. Standardize column names to lowercase snake_case
    .withColumnRenamed("Ticket_ID", "ticket_id")
    .withColumnRenamed("Customer_Name", "customer_name")
    .withColumnRenamed("Customer_Email", "customer_email")
    .withColumnRenamed("Ticket_Subject", "ticket_subject")
    .withColumnRenamed("Ticket_Description", "ticket_description")
    .withColumnRenamed("Issue_Category", "issue_category")
    .withColumnRenamed("Priority_Level", "priority_level")
    .withColumnRenamed("Ticket_Channel", "ticket_channel")
    .withColumnRenamed("Submission_Date", "submission_date")
    .withColumnRenamed("Resolution_Time_Hours", "resolution_time_hours")
    .withColumnRenamed("Assigned_Agent", "assigned_agent")
    .withColumnRenamed("Satisfaction_Score", "satisfaction_score")
    
    # 3. Trim IDs and standardize text fields
    .withColumn("ticket_id", F.trim(F.col("ticket_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("customer_name", F.initcap(F.trim(F.col("customer_name"))))
    .withColumn("customer_email", F.lower(F.trim(F.col("customer_email"))))
    
    # Extract original description up to the first '.' or '?'
    .withColumn(
        "ticket_description",
        F.regexp_extract(F.col("ticket_description"), r"^(.*?[.?])", 1)
    )
    
    .withColumn("issue_category", F.initcap(F.trim(F.col("issue_category"))))
    .withColumn("priority_level", F.initcap(F.trim(F.col("priority_level"))))
    .withColumn("ticket_channel", F.initcap(F.trim(F.col("ticket_channel"))))
    .withColumn("assigned_agent", F.initcap(F.trim(F.col("assigned_agent"))))
    
    # Make null priority levels as Medium (the default priority)
    .withColumn("priority_level",
                F.coalesce(F.col("priority_level"), F.lit("Medium"))
    )
    
    # 4. Handle Submission Date (Default to 'Unknown' string if null)
    .withColumn(
        "submission_date",
        F.coalesce(F.trim(F.col("submission_date")), F.lit("Unknown"))
    )
    
    # 5. Handle null Assigned Agents
    .withColumn(
        "assigned_agent",
        F.coalesce(F.col("assigned_agent"), F.lit("Unknown"))
    )
    
    # 6. Type Casting for Metrics
    .withColumn("resolution_time_hours", F.col("resolution_time_hours").cast(DoubleType()))
    .withColumn("satisfaction_score", F.col("satisfaction_score").cast(IntegerType()))
    
    # 7. DROP ROWS where satisfaction_score is null or outside 1-5
    .filter(
        F.col("satisfaction_score").isNotNull() &
        (F.col("satisfaction_score") >= 1) & 
        (F.col("satisfaction_score") <= 5)
    )
    
    # 8. DROP ROWS where resolution_time_hours is null
    .filter(F.col("resolution_time_hours").isNotNull())
    
    # Deduplicate on ticket primary key
    .dropDuplicates(["ticket_id"])
)

In [56]:
# Inspect null counts per column after cleaning
tickets_silver.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in tickets_silver.columns
]).show()

+---------+-----------+-------------+--------------+--------------+------------------+--------------+--------------+--------------+---------------+---------------------+--------------+------------------+
|ticket_id|customer_id|customer_name|customer_email|ticket_subject|ticket_description|issue_category|priority_level|ticket_channel|submission_date|resolution_time_hours|assigned_agent|satisfaction_score|
+---------+-----------+-------------+--------------+--------------+------------------+--------------+--------------+--------------+---------------+---------------------+--------------+------------------+
|        0|          0|            0|             0|             0|                 0|             0|             0|             0|              0|                    0|             0|                 0|
+---------+-----------+-------------+--------------+--------------+------------------+--------------+--------------+--------------+---------------+---------------------+--------------+

In [63]:
# Usage_Logs
df_ulogs = (
    spark.read
    .format("json")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("hdfs://localhost:9000/bronze/usage")
)


# Print schema and row count
df_ulogs.printSchema()
print("Total Usage Log Records:", df_tickets.count())

# Inspect null counts per column
df_ulogs.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df_ulogs.columns
]).show()

root
 |-- customer_id: string (nullable = true)
 |-- monthly_balance: string (nullable = true)
 |-- num_products: long (nullable = true)
 |-- product_type: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- usage_log_id: string (nullable = true)

Total Usage Log Records: 20000
+-----------+---------------+------------+------------+-------------+---------+------------+
|customer_id|monthly_balance|num_products|product_type|source_system|timestamp|usage_log_id|
+-----------+---------------+------------+------------+-------------+---------+------------+
|        897|            958|         901|         868|          611|        0|           0|
+-----------+---------------+------------+------------+-------------+---------+------------+



In [69]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, TimestampType

usage_logs_silver = (
    df_ulogs
    # 1. Drop rows missing primary key or customer link
    .filter(
        F.col("usage_log_id").isNotNull() & (F.trim(F.col("usage_log_id")) != "") &
        F.col("customer_id").isNotNull() & (F.trim(F.col("customer_id")) != "")
    )
    
    # 2. Trim identifiers
    .withColumn("usage_log_id", F.trim(F.col("usage_log_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    
    # 3. Clean and cast numeric metrics
    .withColumn("monthly_balance", F.col("monthly_balance").cast(DoubleType()))
    .withColumn("num_products", F.col("num_products").cast(IntegerType()))
    
    # Filter out invalid negative balances or product counts
    .filter(
        (F.col("monthly_balance") >= 0) &
        (F.col("num_products") >= 0)
    )
    
    # 4. Standardize text categories
    .withColumn(
        "product_type", 
        F.initcap(F.coalesce(F.trim(F.col("product_type")), F.lit("Unknown")))
    )
    .withColumn(
        "source_system", 
        F.upper(F.coalesce(F.trim(F.col("source_system")), F.lit("UNKNOWN")))
    )
    
    # 5. Parse timestamp and drop missing/unparseable timestamps
    .withColumn(
        "timestamp",
        F.coalesce(
            F.col("timestamp").cast(TimestampType()),
            F.to_timestamp(F.col("timestamp"), "yyyy-MM-dd HH:mm:ss"),
            F.to_timestamp(F.col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss")
        )
    )
    .filter(F.col("timestamp").isNotNull())
    
    # 6. Deduplicate by unique log ID
    .dropDuplicates(["usage_log_id"])
)

In [71]:
usage_logs_silver.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in usage_logs_silver.columns
]).show()

usage_logs_silver

+-----------+---------------+------------+------------+-------------+---------+------------+
|customer_id|monthly_balance|num_products|product_type|source_system|timestamp|usage_log_id|
+-----------+---------------+------------+------------+-------------+---------+------------+
|          0|              0|           0|           0|            0|        0|           0|
+-----------+---------------+------------+------------+-------------+---------+------------+



customer_id,monthly_balance,num_products,product_type,source_system,timestamp,usage_log_id
CUST-100014,45375.4,3,Credit Card,CORE_BANKING,2025-04-23 20:35:...,USG-300011
CUST-100163,13604.5,1,Investment Fund,CORE_BANKING,2026-02-11 06:18:...,USG-300261
CUST-100388,37550.75,4,Mortgage,MOBILE_APP,2026-04-15 03:46:...,USG-300629
CUST-100418,19143.45,1,Savings Account,WEB_PORTAL,2026-07-30 23:46:...,USG-300674
CUST-100471,22741.99,3,Investment Fund,MOBILE_APP,2025-08-10 05:17:...,USG-300753
CUST-100503,21528.35,1,Savings Account,MOBILE_APP,2025-09-22 18:30:...,USG-300801
CUST-101217,46825.33,4,Personal Loan,WEB_PORTAL,2025-11-23 08:19:...,USG-301913
CUST-101271,5653.28,1,Savings Account,MOBILE_APP,2025-10-01 18:32:...,USG-302003
CUST-101663,38222.65,5,Savings Account,MOBILE_APP,2026-04-22 20:59:...,USG-302630
CUST-101784,10413.86,5,Checking Account,CORE_BANKING,2026-05-30 15:58:...,USG-302823


In [74]:
# Write customers_silver
(
    customers_silver.coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .save("hdfs://localhost:9000/silver/customers")
)

# Write tickets_silver
(
    tickets_silver.coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .save("hdfs://localhost:9000/silver/tickets")
)

# Write usage_logs_silver
(
    usage_logs_silver.coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .save("hdfs://localhost:9000/silver/usage_logs")
)